In [1]:
import os
from pathlib import Path
import json
import requests
import re
import math
import ipywidgets as widgets
from IPython.display import display
from dotenv import load_dotenv
from openai import OpenAI
from ollama import chat
from anthropic import Anthropic
from IPython.display import Markdown, display

In [2]:
for _d in [Path.cwd(), *Path.cwd().parents]:
    _env = _d / ".env"
    if _env.is_file():
        load_dotenv(_env, override=True)
        break

API_KEY_WEATHER = os.environ["WEATHERAPI_KEY"]
API_KEY_EXCHANGE = os.environ["EXCHANGE_RATE_API_KEY"]

In [3]:
SYSTEM_PROMPT = """
You are a strict routing assistant.

Your ONLY job is to:
1. Choose exactly ONE tool
2. Extract and normalize parameters for that tool

You must NEVER answer the user.
You must NEVER explain anything.
You must ONLY return valid JSON.

-------------------------
AVAILABLE TOOLS:
-------------------------
1. getWeather(cities, mode)
2. calculateMath(expression)
3. getExchangeRate(fromCurrency, toCurrency, amount optional)
4. generalChat(userInput)

-------------------------
GENERAL RULES:
-------------------------
- The user can write in ANY language (English, Hebrew, Arabic, Spanish, etc.)
- You must ALWAYS detect intent correctly regardless of language
- You must ALWAYS output valid JSON only
- You must NEVER include natural language in the output

-------------------------
WEATHER RULES (CRITICAL):
-------------------------
- If the request is about weather → ALWAYS use getWeather
- If 1 city is mentioned → return one city in "cities"
- If 2 or more cities are mentioned → return ALL cities
- City names MUST be converted to English for API usage
- Never send non-English city names to the tool

Examples:

User: "מה המזג אוויר של חיפה"
→ cities: ["Haifa"]

User: "weather in Paris"
→ cities: ["Paris"]

User: "פי כמה חמה תל אביב ממוסקבה"
→ cities: ["Tel Aviv", "Moscow"]

User: "how much hotter is Tel Aviv than Moscow"
→ cities: ["Tel Aviv", "Moscow"]

IMPORTANT:
- City extraction must be semantic (based on meaning), not language
- Always convert cities to English before returning

-------------------------
COMPARISON MODE RULE (CRITICAL):
-------------------------
You must detect the type of comparison:

- "how much hotter / colder / difference" → mode = "diff"
- "how many times / twice / ratio" → mode = "ratio"
- "percentage difference" → mode = "percent"
- anything else → mode = "compare"

Examples:

User: "how much hotter is Tel Aviv than Moscow"
→ mode: "diff"

User: "how many times hotter is Dubai than London"
→ mode: "ratio"

User: "is Cairo twice as hot as Paris"
→ mode: "ratio"

-------------------------
MATH RULES:
-------------------------
- If input is a math expression → use calculateMath
- Return expression exactly as given

-------------------------
CURRENCY RULES:
-------------------------
- If user provides ONE currency → convert to ILS
- If user provides TWO currencies → convert between them
- If amount is mentioned → include it
- Otherwise omit amount

-------------------------
GENERAL CHAT RULE:
-------------------------
Use generalChat ONLY if:
- No other tool applies
- OR the request is not weather, math, or currency

-------------------------
OUTPUT FORMAT (STRICT):
-------------------------
Return ONLY valid JSON:

Weather example:
{
  "tool": "getWeather",
  "parameters": {
    "cities": ["City1", "City2"],
    "mode": "diff"
  }
}

Math example:
{
  "tool": "calculateMath",
  "parameters": {
    "expression": "2+2"
  }
}

Currency example:
{
  "tool": "getExchangeRate",
  "parameters": {
    "fromCurrency": "USD",
    "toCurrency": "ILS",
    "amount": 100
  }
}

General chat example:
{
  "tool": "generalChat",
  "parameters": {
    "userInput": "..."
  }
}

-------------------------
FINAL RULE:
-------------------------
Return ONLY JSON. No explanations. No extra text.
"""

In [4]:
LOG_FILE = "execution_log.txt"

def log_interaction(user_input, response):
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(f"You: {user_input}\n")
        f.write(f"Bot: {response}\n")
        f.write("\n")

def reset_log():
    if os.path.exists(LOG_FILE):
        os.remove(LOG_FILE)
                

In [5]:
HISTORY_FILE = "history.json"

def load_history():
    if os.path.exists(HISTORY_FILE):
        try:
            with open(HISTORY_FILE, "r", encoding="utf-8") as f:
                history = json.load(f)

            print("ברוך שובך 👋")
            return history

        except Exception:
            print("History file corrupted, starting fresh.")
            return []

    else:
        return []


def save_history(history):
    with open(HISTORY_FILE, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)


def reset_history():
    if os.path.exists(HISTORY_FILE):
        os.remove(HISTORY_FILE)
    return []

In [6]:
load_dotenv(override=True)
client = OpenAI()  

In [7]:
tools = [
    {
        "name": "getWeather",
        "description": "Get current weather for a city",
        "parameters": {
            "city": "string"
        }
    },
    {
        "name": "calculateMath",
        "description": "Calculate a math expression",
        "parameters": {
            "expression": "string"
        }
    },
     {
        "name": "getExchangeRate",
        "description": "Convert currency between two currencies , including questions like 'how much is 100 EUR in USD'",
        "parameters": {
            "fromCurrency": "string",
            "toCurrency": "string",
            "amount": "number"
        }
    },
    {
        "name": "generalChat",
        "description": "General conversation"
    }
]

In [8]:
def calculateMath(expression):
    try:
     
        parts = re.findall(r'[0-9+\-*/().]+', expression)

        if not parts:
            return "Invalid expression"

        clean_expr = "".join(parts)

    
        result = eval(clean_expr, {"__builtins__": None}, {})

        return result

    except Exception:
        return "Invalid expression"

In [9]:
def get_weather_description(code):
    if code == 0:
        return "Clear sky"
    elif code in [1, 2, 3]:
        return "Cloudy"
    elif code in [61, 63, 65]:
        return "Rain"
    elif code in [71, 73, 75]:
        return "Snow"
    else:
        return "Unknown"

In [10]:
def getWeather(city: str):
    if not city:
        return "Please provide a city"

    try:
        geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city}"
        geo_response = requests.get(geo_url).json()

        if "results" not in geo_response or not geo_response["results"]:
            return "City not found"

        lat = geo_response["results"][0]["latitude"]
        lon = geo_response["results"][0]["longitude"]

        weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
        weather_response = requests.get(weather_url).json()

        if "current_weather" not in weather_response:
            return "Weather data unavailable"

        temp = weather_response["current_weather"]["temperature"]
        wind = weather_response["current_weather"]["windspeed"]
        weather_code = weather_response["current_weather"]["weathercode"]

        description = get_weather_description(weather_code)

        return f"{city}: {temp}°C, {description}, wind {wind} km/h"

    except Exception as e:
        return f"Error: {str(e)}"

In [11]:
def getExchangeRate(fromCurrency: str, toCurrency: str = None):
    fromCurrency = fromCurrency.upper()
 
    if not toCurrency:
        toCurrency = "ILS"
    else:
        toCurrency = toCurrency.upper()

    url = f"https://v6.exchangerate-api.com/v6/{API_KEY_EXCHANGE}/latest/{fromCurrency}"

    try:
        response = requests.get(url)
        data = response.json()

        if data.get("result") != "success":
            return "API error"

        rate = data["conversion_rates"].get(toCurrency)

        if rate is None:
            return f"No data for {toCurrency}"

        return f"1 {fromCurrency} = {rate} {toCurrency}"

    except Exception as e:
        return f"Error: {str(e)}"

In [12]:
def extract_amount(text):
    match = re.search(r'\d+(\.\d+)?', text)
    return float(match.group()) if match else None

In [13]:
def generalChat(context, userInput):
    messages = [
        {"role": m["role"], "content": str(m["content"])}
        for m in context
    ] + [
        {"role": "user", "content": str(userInput)}
    ]

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages
    )

    return response.choices[0].message.content

In [14]:
def extract_json(text):
    match = re.search(r'\{.*\}', text, re.DOTALL)
    return match.group(0) if match else "{}"


def openai_router(user_input):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_input}
    ]

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        temperature=0   
       
    )

    raw_output = response.choices[0].message.content
    


    clean_json = extract_json(raw_output)

    try:
        return json.loads(clean_json)
    except:
        return {
            "tool": "generalChat",
            "parameters": {"userInput": user_input}
        }

In [15]:
def extract_cities(user_input):
    # simple rule-based first (you can later replace with LLM)
    cities = []

    # very basic pattern ideas
    # split by "then", "than", "vs", "compared to"
    separators = [" then ", " than ", " vs ", " compared to "]

    text = user_input.lower()
    for sep in separators:
        if sep in text:
            parts = text.split(sep)
            cities = [parts[0], parts[1]]
            break

    return [c.strip() for c in cities]

In [16]:
def extract_temp(weather_result):
    nums = re.findall(r'[-+]?\d*\.?\d+', str(weather_result))

    if not nums:
        return None

    return float(nums[0])

In [17]:
def chat(user_input, history):

    try:
        route = openai_router(user_input)
    except Exception:
        route = {"tool": "generalChat", "parameters": {}}

    tool = route.get("tool", "generalChat")
    params = route.get("parameters", {})

    # 🔹 WEATHER
    if tool == "getWeather":
        cities = params.get("cities", [])

        # safety check
        if not cities :
            result = "Missing city"
        
        elif len(cities) == 1:
            weather = getWeather(cities[0])
            result = weather

        else:
            mode = params.get("mode", "diff")
            w1 = getWeather(cities[0])
            w2 = getWeather(cities[1])

            # extract temperature (adjust keys if needed)
            t1 = extract_temp(w1)
            t2 = extract_temp(w2)

            if t1 is None or t2 is None:
                result = "Weather data unavailable for comparison"
                return result

            diff = abs(t1 - t2)
            diff = abs(t1 - t2)
            ratio = max(t1, t2) / min(t1, t2) if min(t1, t2) != 0 else None
            percent = (diff / min(t1, t2)) * 100 if min(t1, t2) != 0 else None
            hotter = cities[0] if t1 > t2 else cities[1]
            colder = cities[1] if hotter == cities[0] else cities[0]

            if mode == "diff":
                result = f"{hotter} is hotter than {colder} by {diff:.1f}°C"

            elif mode == "ratio":
                if ratio is None:
                    result = "Cannot compute ratio"
                else:
                    result = f"{hotter} is about {ratio:.1f} times hotter than {colder}"

            elif mode == "percent":
                result = f"{hotter} is about {percent:.1f}% hotter than {colder}"

            else:
                result = f"{hotter} is hotter than {colder}"

    # 🔹 MATH
    elif tool == "calculateMath":
        expr = params.get("expression")
        result = calculateMath(expr) if expr else "Missing expression"

    # 🔹 EXCHANGE RATE
    elif tool == "getExchangeRate":
        from_curr = params.get("fromCurrency")
        to_curr = params.get("toCurrency")
        amount = params.get("amount")

        if not from_curr:
            result = "Missing currency"

        else:
            
            if not to_curr:
                to_curr = "ILS"

            
            if not amount:
                amount = extract_amount(user_input)

            rate_result = getExchangeRate(from_curr, to_curr)

            try:
                numbers = re.findall(r'[-+]?\d*\.?\d+', rate_result)

                if len(numbers) < 2:
                    result = rate_result
                else:
                    rate = float(numbers[1])

                    if amount:
                        converted = round(amount * rate, 2)
                        result = f"{amount} {from_curr} = {converted} {to_curr}"
                    else:
                        result = rate_result

            except Exception:
                result = rate_result

    # 🔹 FALLBACK
    else:
        result = generalChat(history, user_input)

    # 🔹 SAVE HISTORY
    history.append({"role": "user", "content": user_input})
    history.append({"role": "assistant", "content": str(result)})
    log_interaction(user_input, result)
    save_history(history)

    return result

In [18]:
def main():
    history = load_history()

    input_box = widgets.Text(
        placeholder='Type your message...',
        description='You:',
        layout=widgets.Layout(width='70%')
    )

    output_box = widgets.Output()

 
    with output_box:
        if history:
            print("Previous conversation 👇\n")
        for message in history:
            role = message["role"]
            content = message["content"]

            if role == "user":
                print(f"You: {content}")
            else:
                print(f"Bot: {content}")
            print("")

    def on_submit(change):
        nonlocal history

        user_input = input_box.value   

        if not user_input.strip():
            return

        if not user_input.strip():
            return

        if user_input.lower() == "/exit":
            with output_box:
                print("Goodbye 👋")
            return

        if user_input.strip().lower() == "/reset":
            history = reset_history()
            output_box.clear_output()
            with output_box:
                print("History cleared 🧹")
            input_box.value = ""
            return

        response = chat(user_input, history)

        with output_box:
            
            print(f"You: {user_input}")
            print(f"Bot: {response}")
            print("")

        input_box.value = ""

    input_box.on_submit(on_submit)

    display(input_box, output_box)

In [ ]:
main()

ברוך שובך 👋


C:\Users\dorno\AppData\Local\Temp\ipykernel_20472\547648382.py:60: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  input_box.on_submit(on_submit)


Text(value='', description='You:', layout=Layout(width='70%'), placeholder='Type your message...')

Output()